# CS406 · SAGE Failure Analysis (Kaggle Fixed) trên FMB `scene_v1` (Train-only · Kaggle path/index FIXED)

**Dành cho nhiệm vụ Lộc bé trước 20/11/2026.** Notebook này phục vụ **phân tích lỗi và chọn 20–30 ví dụ VIS/IR/SAGE**; không fine-tune, không chạy segmentation B3, không sử dụng Test hay 228 Dev để lựa chọn giải pháp. Chỉ sử dụng nhãn Train để *phân tích lỗi*, không làm tín hiệu inference. Giữ nguyên baseline đã khóa của nhóm.

Dữ liệu bắt buộc: `data-FMB` (ảnh Train VIS/IR/Label), `q02b_kaggle_bundle` (manifest scene_v1, file lock) và ảnh SAGE fusion **của chính 992 Train IDs**, dự kiến trong `sage-q02c-frozen-input`. `train_probe_subset.csv` chỉ là nguồn **phụ**, phải kiểm tra nó là subset của Train. `vis-dev`/`sage-dev` là output của Dev: **không dùng để chọn hướng**. `nvidia-mit-b3-pinned` dành cho downstream đánh giá, không cần GPU trong notebook này.

*Lưu ý*: Input names trong Kaggle có thể khác nhau. Cell phát hiện ảnh SAGE sẽ dừng an toàn nếu chưa xác định được đường dẫn/contract, thay vì tự chọn nhầm ảnh Dev hoặc ảnh VIS.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv, io, os, zipfile, json, hashlib, math, random, re, statistics
import numpy as np
import cv2
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

INPUT = Path('/kaggle/input')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT = Path('/kaggle/working/SAGE-research/locbe_failure_analysis') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
SAGE_SOURCE_OVERRIDE = Path('/kaggle/input/datasets/eoquiither/q02b-data/sage-q02c-frozen-input/q02b_runs/images/sage/train')  # Ảnh SAGE Train của Quân, không dùng Dev/Test
FMB_TRAIN_OVERRIDE = None    # Folder trực tiếp chứa Visible/Infrared/Label của official Train
MAX_ANALYSIS_IMAGES = None  # None = toàn bộ ảnh Train có SAGE fusion. Test/Dev luôn bị loại.
PERSON_ID = 8   # FMB: person=8, car=9, truck=10, bus=11, motorcycle=12; verify against locked B3 class names
SEED = 20261118
print('RUN_ID:', RUN_ID, 'OUTPUT:', OUT)
print('CV2:', cv2.__version__, 'Pandas:', pd.__version__)

In [ ]:
# Dataset paths xác minh từ Kaggle notebook CS406_1811 (19/11/2026).
# Không sử dụng FMB Test hoặc Sage-dev. Có thể thay lại các path khi dataset slug đổi.
BASE = Path('/kaggle/input/datasets/eoquiither')
BUNDLE_MOUNT = BASE / 'q02b-data' / 'q02b_kaggle_bundle'
FMB_MOUNT = BASE / 'fmb-data' / 'FMB'
FROZEN_MOUNT = BASE / 'q02b-data' / 'sage-q02c-frozen-input'
PROBE_MOUNT = None  # Train-probe chưa được gắn, optional.

for name, p in [('Q02B BUNDLE', BUNDLE_MOUNT), ('FMB', FMB_MOUNT), ('SAGE FROZEN', FROZEN_MOUNT)]:
    if not p.is_dir():
        raise FileNotFoundError(f'{name} không tồn tại: {p}; kiểm tra Kaggle Add Input')
    print(name, '->', p)
print('MOUNTS: PASS')

In [ ]:
# Kiểm chứng ZIP Q02B; chỉ trích xuất bundle nhỏ, không trích xuất Test hay archive ảnh lớn.
def locate_bundle(m):
    if (m/'q02b_lock.json').is_file(): return m
    dirs=[p for p in m.rglob('q02b_lock.json') if len(p.relative_to(m).parts) <= 6]
    if len(dirs)==1: return dirs[0].parent
    zips=[p for p in m.rglob('*.zip') if 'q02b' in p.name.lower()]
    if len(zips)!=1: raise RuntimeError(f'Không xác định được chính xác Q02B ZIP: {zips}')
    target=OUT/'q02b_verified'; target.mkdir()
    with zipfile.ZipFile(zips[0]) as z:
        for info in z.infolist():
            dest=(target/info.filename).resolve()
            if not dest.is_relative_to(target.resolve()): raise RuntimeError('Unsafe bundle path')
            if not info.is_dir():
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(z.read(info))
    return target

Q02B=locate_bundle(BUNDLE_MOUNT)
lock=json.loads((Q02B/'q02b_lock.json').read_text(encoding='utf-8'))
for rel, expected in lock['sha256'].items():
    q=Q02B/rel
    assert q.is_file() and hashlib.sha256(q.read_bytes()).hexdigest()==expected, f'Q02B lock mismatch: {rel}'
def read_manifest(split):
    with (Q02B/'manifests'/'scene_v1'/f'{split}.csv').open(encoding='utf-8-sig',newline='') as f:
        return list(csv.DictReader(f))
TRAIN=read_manifest('train'); DEV=read_manifest('dev')
TRAIN_IDS={r['image_id'] for r in TRAIN}; DEV_IDS={r['image_id'] for r in DEV}
assert len(TRAIN)==len(TRAIN_IDS)==992 and len(DEV)==len(DEV_IDS)==228
assert TRAIN_IDS.isdisjoint(DEV_IDS)
assert all(r['split']=='train' for r in TRAIN) and all(r['split']=='dev' for r in DEV)
config=json.loads((Q02B/'manifests'/'scene_v1'/'provenance.json').read_text())
print('Q02B PIN VERIFY: PASS /',len(lock['sha256']),'files')
print('scene_v1: Train=',len(TRAIN),' Dev=',len(DEV),' disjoint: PASS')
print('moved_count:',config.get('moved_count'))

In [ ]:
# Tìm đúng official TRAIN (Visible, Infrared, Label). Không đi vào bất kỳ thư mục Test nào.
def find_train_folder(base):
    if FMB_TRAIN_OVERRIDE is not None:
        p=Path(FMB_TRAIN_OVERRIDE)
        if all((p/m).is_dir() for m in ('Visible','Infrared','Label')): return p
        raise RuntimeError('FMB_TRAIN_OVERRIDE thiếu Visible/Infrared/Label')
    matches=[]
    for root,dirs,files in os.walk(base):
        depth=len(Path(root).relative_to(base).parts)
        dirs[:]=[d for d in dirs if d.lower() not in {'test','fmbtest','.git'} and depth < 6]
        p=Path(root)
        if p.name.lower()=='train' and all((p/m).is_dir() for m in ('Visible','Infrared','Label')):
            matches.append(p)
    if len(matches)!=1: raise RuntimeError(f'Phải xác định duy nhất FMB Train folder: {matches}. Điền FMB_TRAIN_OVERRIDE.')
    return matches[0]
FMB_TRAIN=find_train_folder(FMB_MOUNT)
print('OFFICIAL FMB TRAIN ROOT:',FMB_TRAIN)

# Check checksum subset để bảo đảm đúng dataset release.
for r in TRAIN[:10]:
    sid=r['image_id']
    for folder,key in [('Visible','vis_md5'),('Infrared','ir_md5'),('Label','label_md5')]:
        f=FMB_TRAIN/folder/(sid+'.png')
        assert f.is_file() and hashlib.md5(f.read_bytes()).hexdigest()==r[key], (sid, folder)
print('Sample Train MD5 (10 x 3): PASS')

In [ ]:
# Tìm optional Train-probe (chỉ metadata; tuyệt đối không đưa Dev/Test vào pool).
PROBE_IDS=set()
if PROBE_MOUNT is not None:
    candidates=[]
    for root,dirs,files in os.walk(PROBE_MOUNT):
        depth=len(Path(root).relative_to(PROBE_MOUNT).parts)
        dirs[:]=[d for d in dirs if 'test' not in d.lower() and depth<4]
        candidates.extend(Path(root)/name for name in files if name.lower().endswith('.csv') and ('probe' in name.lower() or 'subset' in name.lower()))
    print('Probe files:',[str(x) for x in candidates[:10]])
    if len(candidates)==1:
        dfp=pd.read_csv(candidates[0],dtype=str)
        field=next((col for col in dfp.columns if col.lower().strip() in ('image_id','id','filename')), None)
        if field:
            ids={str(x).split('.')[0].zfill(5) for x in dfp[field].dropna()}
            if ids <= TRAIN_IDS:
                PROBE_IDS=ids
                print('Train-probe verified subset:',len(PROBE_IDS),'IDs')
            else:
                print('WARNING: Train-probe contains IDs outside Train; IGNORING it. Outside:',sorted(ids-TRAIN_IDS)[:20])
        else: print('WARNING: Cannot parse probe IDs. Columns:',list(dfp))
    elif len(candidates)>1: print('WARNING: Probe CSV ambiguous, not used until confirmed')
else:
    print('Train-probe dataset not mounted. Proceed with scene_v1 Train only.')

In [ ]:
# SAGE PNG index — trực tiếp theo 992 Train IDs; không dùng regex hoặc quét Dev/Test.
# Chỉ nhận ảnh Train nằm trong thư mục fused output đã xác minh.
SOURCE = Path(SAGE_SOURCE_OVERRIDE) if SAGE_SOURCE_OVERRIDE is not None else FROZEN_MOUNT / 'q02b_runs/images/sage/train'
SOURCE = SOURCE.resolve()
if not SOURCE.is_dir():
    raise FileNotFoundError(f'Không thấy thư mục SAGE Train: {SOURCE}')
if SOURCE.name.lower() != 'train' or SOURCE.parent.name.lower() != 'sage':
    raise RuntimeError(f'SOURCE không trỏ vào thư mục images/sage/train: {SOURCE}')
if any(x.lower() in {'test', 'dev'} for x in SOURCE.parts):
    raise PermissionError('Không dùng SAGE Dev/Test để phân tích Train')

# Đối chiếu thẳng từng image_id của scene_v1, không tìm theo tên mở rộng bất kỳ.
missing = sorted(sid for sid in TRAIN_IDS if not (SOURCE / f'{sid}.png').is_file())
if missing:
    print('Missing examples:', missing[:20])
    raise RuntimeError(f'Thiếu {len(missing)} ảnh SAGE tương ứng scene_v1 Train')
SAGE_INDEX = {sid: ('file', SOURCE / f'{sid}.png', None) for sid in sorted(TRAIN_IDS)}
assert len(SAGE_INDEX) == 992 and not (set(SAGE_INDEX) & DEV_IDS)

# Chỉ mở 1 ảnh Train để xác nhận decoding trước khi xử lý toàn bộ.
first_id = sorted(TRAIN_IDS)[0]
first = cv2.imread(str(SAGE_INDEX[first_id][1]), cv2.IMREAD_COLOR)
if first is None or first.shape != (600, 800, 3) or first.dtype != np.uint8:
    raise RuntimeError(f'Ảnh SAGE {first_id} không đúng PNG/BGR uint8 600x800: {None if first is None else (first.shape, first.dtype)}')
print('SAGE input coverage:', len(SAGE_INDEX), 'of', len(TRAIN_IDS), 'Train IDs; ambiguous: 0')
print('SAGE sample:', first_id, SAGE_INDEX[first_id][1])
print('SAGE TRAIN INDEX: PASS')

In [ ]:
# Đọc từng ảnh theo manifest; giữ ảnh gốc, không sửa SAGE. Chỉ phân tích, không training.
zip_cache={}
def read_sage(sid):
    kind,path,member=SAGE_INDEX[sid]
    if kind=='file': image=cv2.imread(str(path),cv2.IMREAD_COLOR)
    else:
        key=str(path)
        if key not in zip_cache: zip_cache[key]=zipfile.ZipFile(path)
        image=cv2.imdecode(np.frombuffer(zip_cache[key].read(member),dtype=np.uint8),cv2.IMREAD_COLOR)
    if image is None: raise RuntimeError('Cannot decode fused SAGE '+sid)
    return image

def read_triplet(sid):
    vis=cv2.imread(str(FMB_TRAIN/'Visible'/(sid+'.png')),cv2.IMREAD_COLOR)
    ir=cv2.imread(str(FMB_TRAIN/'Infrared'/(sid+'.png')),cv2.IMREAD_COLOR)
    sage=read_sage(sid)
    if any(img is None or img.shape!=(600,800,3) or img.dtype!=np.uint8 for img in (vis,ir,sage)):
        raise ValueError(f'Unexpected shape/color contract for {sid}')
    label=np.asarray(Image.open(FMB_TRAIN/'Label'/(sid+'.png')))
    if label.ndim!=2 or label.shape!=(600,800) or np.max(label)>14:
        raise ValueError(f'Label not indexed 0..14: {sid}; shape={label.shape}')
    return vis,ir,sage,label

def sobel_energy(y):
    a=y.astype('float32')/255.0
    sx=cv2.Sobel(a,cv2.CV_32F,1,0,ksize=3)
    sy=cv2.Sobel(a,cv2.CV_32F,0,1,ksize=3)
    return np.sqrt(sx*sx+sy*sy)

def thermal_contrast(obj, mask):
    mask=(mask>0).astype('uint8')
    if int(mask.sum())<40: return float('nan')
    ring=cv2.dilate(mask,np.ones((15,15),'uint8'))-mask
    if int(ring.sum())<40: return float('nan')
    return abs(float(np.median(obj[mask>0]))-float(np.median(obj[ring>0])))

def features(r):
    sid=r['image_id']
    vis,ir,sage,gt=read_triplet(sid)
    Yv=cv2.cvtColor(vis,cv2.COLOR_BGR2YCrCb)[:,:,0]
    Yi=cv2.cvtColor(ir,cv2.COLOR_BGR2GRAY)
    Yf=cv2.cvtColor(sage,cv2.COLOR_BGR2YCrCb)[:,:,0]
    Gv=sobel_energy(Yv); Gf=sobel_energy(Yf)
    edge_mask=Gv>np.percentile(Gv,80)
    edge_ratio=float(np.mean(Gf[edge_mask])/(np.mean(Gv[edge_mask])+1e-8))
    m_person=(gt==PERSON_ID).astype('uint8')
    m_vehicle=np.isin(gt,[9,10,11,12]).astype('uint8')
    ir_person=thermal_contrast(Yi,m_person)
    sage_person=thermal_contrast(Yf,m_person)
    ir_vehicle=thermal_contrast(Yi,m_vehicle)
    sage_vehicle=thermal_contrast(Yf,m_vehicle)
    return dict(image_id=sid,mean_vis_y=float(Yv.mean()),dark_fraction=float((Yv<64).mean()),
                edge_retention=edge_ratio,vis_edge_mean=float(Gv.mean()),fused_edge_mean=float(Gf.mean()),
                ir_person_contrast=ir_person,sage_person_contrast=sage_person,
                person_ir_retention=float(sage_person/(ir_person+1e-6)) if np.isfinite(ir_person) and ir_person>8 else np.nan,
                ir_vehicle_contrast=ir_vehicle,sage_vehicle_contrast=sage_vehicle,
                vehicle_ir_retention=float(sage_vehicle/(ir_vehicle+1e-6)) if np.isfinite(ir_vehicle) and ir_vehicle>8 else np.nan,
                person_pixels=int(m_person.sum()),vehicle_pixels=int(m_vehicle.sum()),
                source_contract='SAGE PNG read as OpenCV BGR -> YCrCb Y; raw source uint8',
                label_role='Train analysis only; NEVER inference supervision')

rows=[r for r in TRAIN if r['image_id'] in SAGE_INDEX]
if MAX_ANALYSIS_IMAGES:
    rows=rows[:int(MAX_ANALYSIS_IMAGES)]
print('Analyzing',len(rows),'Train-only image IDs; no Dev/Test access')
results=[]
for index,r in enumerate(rows,1):
    results.append(features(r))
    if index%50==0: print('processed',index,'/',len(rows))
frame=pd.DataFrame(results)
assert set(frame.image_id)<=TRAIN_IDS and not set(frame.image_id)&DEV_IDS
frame.to_csv(OUT/'train_per_image_metrics.csv',index=False)
for z in zip_cache.values(): z.close()
zip_cache.clear()
print('Per-image analysis saved:',OUT/'train_per_image_metrics.csv')
print(frame[['mean_vis_y','edge_retention','person_ir_retention','vehicle_ir_retention']].describe().round(3).to_string())

In [ ]:
# Chọn ví dụ từ Train theo tiêu chí ĐĂNG KÝ TRƯỚC; không chọn theo mIoU Dev.
# Nhãn Train dùng phân nhóm phân tích; masks không bao giờ truyền cho fusion model.
chosen={}; used=set()
def take(name, subset, n):
    count=0
    for sid in subset.image_id.astype(str):
        if sid not in used:
            used.add(sid); chosen[sid]=name; count+=1
        if count>=n: break
    print(name,':',count)

# Dấu hiệu nghi ngờ: không tự gọi đây là lỗi đã được chứng minh.
take('candidate_vis_edge_loss',frame.query('vis_edge_mean > 0.02').sort_values('edge_retention'),8)
take('candidate_person_thermal_attenuation',frame.query('ir_person_contrast >= 12 and person_pixels >= 80').sort_values('person_ir_retention'),8)
take('candidate_vehicle_thermal_attenuation',frame.query('ir_vehicle_contrast >= 12 and vehicle_pixels >= 80').sort_values('vehicle_ir_retention'),6)
take('dark_scene_controls',frame.sort_values('mean_vis_y'),4)
remaining=[sid for sid in frame.image_id.astype(str) if sid not in used]
random.Random(SEED).shuffle(remaining)
for sid in remaining[:max(0,30-len(chosen))]: chosen[sid]='random_train_control'
assert set(chosen)<=TRAIN_IDS and len(chosen)<=30
selected=frame[frame.image_id.isin(chosen)].copy()
selected.insert(1,'selection_reason',selected.image_id.map(chosen))
selected=selected.sort_values(['selection_reason','image_id'])
selected.to_csv(OUT/'selected_20_30_train_cases.csv',index=False)
print('Selected',len(selected),'Train-only cases; reasons:',selected.selection_reason.value_counts().to_dict())

In [ ]:
# Tạo contact sheets VIS | IR | SAGE | Label từ dữ liệu Train (không dùng cho training).
# Ảnh thực nghiệm được lấy trên Kaggle; phiên này chỉ xây code, không giả lập kết quả.
COLORS=[(30,30,30),(80,130,190),(130,130,130),(190,190,130),(180,70,80),
        (80,200,200),(30,150,70),(200,200,230),(230,70,90),(40,110,190),
        (80,160,240),(50,60,200),(120,190,240),(160,60,90),(130,50,180)]
def class_vis(label):
    pal=np.asarray(COLORS,dtype='uint8')
    return Image.fromarray(pal[label],mode='RGB')

def contact_sheet(ids, dest, tile_w=320):
    cell_h=int(tile_w*600/800)
    head_h=43
    canvas=Image.new('RGB',(tile_w*4,(cell_h+head_h)*len(ids)),(245,245,245))
    draw=ImageDraw.Draw(canvas)
    for row,sid in enumerate(ids):
        vis,ir,sage,lab=read_triplet(sid)
        visuals=[Image.fromarray(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB)),
                 Image.fromarray(cv2.cvtColor(ir,cv2.COLOR_BGR2RGB)),
                 Image.fromarray(cv2.cvtColor(sage,cv2.COLOR_BGR2RGB)),class_vis(lab)]
        for c,(name,im) in enumerate(zip(('VIS','IR','SAGE','GT (Train only)'),visuals)):
            y0=row*(head_h+cell_h)
            draw.text((c*tile_w+8,y0+6),f'{sid} | {name}',fill=(15,15,15))
            canvas.paste(im.resize((tile_w,cell_h),Image.Resampling.BILINEAR if c<3 else Image.Resampling.NEAREST), (c*tile_w,y0+head_h))
    canvas.save(dest,optimize=True)

ids=list(selected.image_id.astype(str))
for start in range(0,len(ids),5):
    part=ids[start:start+5]
    contact_sheet(part,OUT/f'contact_sheet_{start//5+1:02d}.png')
print('Contact sheets:',len(list(OUT.glob('contact_sheet_*.png'))))

In [ ]:
# Tạo báo cáo minh bạch: đề xuất giả thuyết, KHÔNG tuyên bố giải pháp đã cải thiện mIoU.
summary={
  'source_split':'scene_v1 Train-only',
  'train_manifest_count':992,'dev_manifest_count':228,
  'analyzed_count':len(frame),'image_ids':list(frame.image_id.astype(str)),
  'selected_count':len(selected),'selected_ids':list(selected.image_id.astype(str)),
  'source':'q02b_kaggle_bundle + FMB official Train + frozen SAGE Train PNGs',
  'trained_new_model':False,'evaluated_dev':False,'evaluated_test':False,
  'notes':'Gradient/contrast metrics descriptive, not evidence of semantic mIoU improvement; manual visual review needed.'
}
(OUT/'analysis_provenance.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
def m(x): return 'NA' if x.dropna().empty else f'{x.median():.3f}'
text=f"""# Failure Analysis — SAGE / FMB scene_v1 (Train-only)

- Run ID: `{RUN_ID}`.
- Nguồn: FMB VIS–IR–Label official Train + Q02B scene_v1 + frozen SAGE Train PNG.
- 992 Train IDs, 228 Dev IDs; chỉ đọc {len(frame)} Train pairs có SAGE. Không dùng Dev/Test.
- Đã khóa 21 SHA256 bundle; kiểm MD5 10 mẫu Train đầu tiên.
- Xuất {len(selected)} trường hợp nghi vấn theo tiêu chí có thể tái lập; **chưa phải lỗi định lượng segmentation đã chứng minh**.

## Tổng quan chỉ số mô tả (median trên tập đã phân tích)
- Mean VIS luminance: {m(frame.mean_vis_y)} (đơn vị 0–255).
- Strong VIS edge retention = mean(Sobel(SAGE)|top20%-VIS-edge)/mean(Sobel(VIS)|top20%-VIS-edge): {m(frame.edge_retention)}.
- Thermal contrast retention ở vùng GT person (Train-only): {m(frame.person_ir_retention)}.
- Thermal contrast retention ở vùng GT vehicle (Train-only): {m(frame.vehicle_ir_retention)}.

## Hai giả thuyết có thể bác bỏ (cần Quân kiểm bằng classwise IoU)
1. H1 — SAGE có thể suy giảm các cấu trúc biên/chi tiết VIS ở một số cảnh, làm tổn hại phân đoạn lớp phụ thuộc biên.
   Bằng chứng cần: vùng có edge retention thấp phải trùng với suy giảm IoU theo lớp trong phân tích Train/Dev đúng protocol; kiểm đối chứng ảnh lỗi/ảnh không lỗi; không chọn gate theo Dev.
   Thử nghiệm tương lai: residual high-frequency VIS được giới hạn theo cue cạnh tại inference, so với zero-residual / fixed mixing, và kiểm trade-off thermal target.
2. H2 — Ở một số đối tượng nóng trong cảnh tối, luminance SAGE có thể làm suy giảm tương phản mục tiêu-vùng nền so với IR, đặc biệt tại người/xe.
   Bằng chứng cần: vùng người/xe có thermal contrast retention thấp đi cùng suy giảm IoU (đối chiếu GT chỉ trong phân tích và training); kiểm trường hợp IR không có lợi.
   Thử nghiệm tương lai: low-frequency IR injection/gate theo độ tin cậy cục bộ từ VIS/IR, không dùng GT lúc inference, kèm các đối chứng TB, fixed IR, global gate đã thử trước đó.

## Bộ ví dụ (ảnh Train-only)
Xem `selected_20_30_train_cases.csv` và `contact_sheet_*.png`.
Các cột `selection_reason`, `edge_retention`, `person_ir_retention`, `vehicle_ir_retention` mô tả tín hiệu quan sát, KHÔNG tự kết luận nguyên nhân của lỗi segmentation.

## Giới hạn
- Chỉ số pixel không suy ra mIoU; không khẳng định novelty. Phải xem ảnh và đối chiếu confusion/predictions của Quân.
- Frozen SAGE PNG source contract phải được kiểm đối chiếu pipeline B3; ảnh source khác contract không được trộn chung khi so điểm.
- Train-probe (nếu có) chỉ là tập con được kiểm ID; nếu được dùng để lọc trước thì phải ghi sai lệch chọn mẫu.
- Nhóm khóa giả thuyết và ablation trên Train; chỉ sau đó mới đánh giá Dev 228 một lần theo protocol đã đăng ký.
"""
(OUT/'failure_analysis.md').write_text(text,encoding='utf-8')
print('REPORT:',OUT/'failure_analysis.md')
print('FILES:',[p.name for p in sorted(OUT.iterdir()) if p.is_file()][:35])

In [ ]:
# ZIP chỉ đầu ra phân tích, không đính kèm dữ liệu ảnh gốc FMB/weights.
from shutil import make_archive
archive=make_archive(str(OUT), 'zip', root_dir=OUT)
print('Evidence archive:',archive)
print('Không sửa GitHub, không train, không inference B3, không đọc Dev/Test.')